# Chapter 7: Planning and Reflection

Companion notebook for *Build an AI Agent (From Scratch)*, Chapter 7.
Each listing in the book maps 1:1 to a cell below.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

## 7.1 Giving agents time to think

Section 7.1 (7.1.1 The limitations of ReAct, 7.1.2 How human experts work, 7.1.3 Why time to think matters) is prose only — no listings. Continue to 7.2 below.


## 7.2 Planning: Setting direction

### 7.2.2 Implementing the planning tool

#### Task data structure


**Listing 7.1** Task data structure definition

In [ ]:
from typing import Literal, List
from pydantic import BaseModel

class Task(BaseModel):
    content: str
    status: Literal["pending", "in_progress", "completed"]

    def __str__(self):
        if self.status == "pending":
            return f"[ ] {self.content}"
        elif self.status == "in_progress":
            return f"[>] **{self.content}**"
        elif self.status == "completed":
            return f"[x] ~~{self.content}~~"
        return self.content

#### The create_tasks tool

**Listing 7.2** Planning tool implementation

In [ ]:
from typing import List
from scratch_agents.tools import tool

@tool
def create_tasks(tasks: List[Task]) -> str:
    """Create or update a task plan.

    WHEN TO USE:
    - Complex queries requiring multiple steps of research
    - Questions that need to combine information from different sources

    WHEN NOT TO USE:
    - Simple questions answerable with a single search
    - Tasks with obvious, straightforward procedures

    HOW TO USE:
    - Regenerate the entire task list with updated statuses
    - Mark completed tasks as 'completed'
    - Mark the next task to work on as 'in_progress'
    - Keep future tasks as 'pending'
    """
    result = []
    for task in tasks:
        result.append(str(task))
    return "\n".join(result)

### 7.2.3 Planning tool usage example

**Listing 7.3** Running an agent with the Planning tool

In [ ]:
from scratch_agents import Agent, LlmClient
from scratch_agents.tools import search_web

agent = Agent(
    model=LlmClient(model="gpt-5-mini"),
    tools=[create_tasks, search_web],
    instructions="You are a helpful assistant that plans before acting on complex queries.",
    max_steps=10,
)

result = await agent.run(
    "If Eliud Kipchoge could maintain his marathon pace indefinitely, "
    "How many thousand hours would it take him to reach the Moon?"
)
print(result.output)

## 7.3 Reflection: Checking and correcting

### 7.3.2 Implementing the reflection tool

#### The reflection tool


**Listing 7.4** Reflection tool implementation

In [ ]:
from scratch_agents.tools import tool

@tool
def reflection(analysis: str, need_replan: bool = False) -> str:
    """Pause and analyze progress before continuing.

    WHEN TO USE:
    1. PROGRESS REVIEW - After completing a meaningful step
       "Kipchoge's record found: 2:01:09. Moving to moon distance research."

    2. ERROR ANALYSIS - When a tool fails or returns unexpected results
       "Wikipedia tool failed. Cause: service unavailable. Alternative: use web search."

    3. RESULT SYNTHESIS - When combining information from multiple sources
       "Two different moon distances found. Problem asks for closest approach, so using perigee: 356,500km."

    4. SELF CHECK - Before providing final answer
       "Have all required data: marathon pace 20.81km/h, moon distance 356,500km. Ready to calculate."

    WHEN NOT TO USE:
    - After every single tool call (excessive overhead)
    - During simple, straightforward operations
    - When everything is proceeding as expected

    Args:
        analysis: Your assessment of current situation and next direction
        need_replan: Set True if the current plan needs modification
    """
    if need_replan:
        return f"Reflection recorded (REPLAN NEEDED): {analysis}"
    return f"Reflection recorded: {analysis}"

### 7.3.3 The real value of reflection: Failure recovery

#### Creating a failing tool


**Listing 7.5** A Wikipedia tool that always fails

In [ ]:
@tool
def get_wikipedia_page(title: str) -> str:
    """Get a Wikipedia page by title."""
    raise RuntimeError("Wikipedia service is temporarily unavailable")

#### Running with reflection

**Listing 7.6** Running an agent with reflection for error recovery

In [ ]:
agent = Agent(
    model=LlmClient(model="gpt-5-mini"),
    tools=[get_wikipedia_page, search_web, reflection],
    instructions="Always try Wikipedia first when researching topics.",
    max_steps=5,
)

result = await agent.run(
    "What is the distance from Earth to Moon at perigee?"
)
print(result.output)

### 7.3.4 Running an agent that uses reflection for research synthesis

**Listing 7.7** Running an agent with reflection for research synthesis

In [ ]:
agent = Agent(
    model=LlmClient(model="gpt-5-mini"),
    tools=[search_web, reflection],
    instructions="You are a research assistant.",
)

result = await agent.run(
    "Research recent developments in quantum computing."
)
print(result.output)

## 7.4 Integrating planning and reflection

### 7.4.1 Failure modes and solutions

This section is prose only — no listings. It revisits the four ReAct failure modes from 7.1.1 and maps each to planning, reflection, or both. See the book for the full mapping table (Table 7.1).
